## Capstone Project of RML (Group 4) - HMDA dataset 

### Pre-modeling 

#### 1.	Data inspection

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd

# Load local dataset
file_path = "2024_lar.txt.csv"
raw_data = pd.read_csv(file_path, sep=",")

# Number of rows
raw_data.shape[0]

In [ ]:
raw_data.head(10)

In [ ]:
list(raw_data.columns)

In [ ]:
raw_data.info()

#### 2.	Target construction

The target variable comes from `action_taken`. We map values 1 and 2 (loan originated or approved but not accepted) to label 1 (approved), and value 3 (denied) to label 0. Everything else — withdrawn applications, files closed for incompleteness, purchased loans — gets dropped, since those outcomes don't tell us whether a credit decision was made.

In [ ]:
import numpy as np

# Create a copy of the original dataset to avoid modifying raw data
df = raw_data.copy()

# Step 1: Filter the dataset to keep only relevant action_taken values
# Keep:
# 1 = Loan originated
# 2 = Approved but not accepted
# 3 = Denied
# Remove all other categories (e.g., withdrawn, incomplete, etc.)
df = df[df["action_taken"].isin([1, 2, 3])]

# Step 2: Create a binary target variable
# Approved (1, 2) → 1
# Denied (3) → 0
df["target"] = np.where(
    df["action_taken"].isin([1, 2]),  # approved cases
    1,
    0  # denied cases
)

# Step 3: Check the distribution of the target variable
# This helps us understand class balance (approved vs denied)
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True))


About 75.7% of applications are approved and 24.3% are denied. The imbalance matters because a model that always predicts "approve" would hit ~76% accuracy without learning anything useful. We'll track FPR and FNR by demographic group rather than relying on accuracy alone to catch this.

#### 3. Data cleaning

**Converting values into `NaN`**

Strings like `"NA"` and `"Exempt"` look like missing data but Python reads them as valid text. Left as-is, they'd get treated as real categories in groupby operations and as separate levels during one-hot encoding. We replace them with `NaN` so pandas handles them correctly throughout the rest of the pipeline.

In [ ]:
df = df.replace(["NA", "Exempt"], np.nan)

**Identifying Missing Values**

We sort columns by their missing-value share to spot the worst offenders. Some columns are almost entirely empty — those won't contribute anything useful to modeling.

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing.head(20)

**Dropping Columns with More Than 90% Missing Values**

Columns where more than 90% of rows are missing aren't worth imputing — they'd mostly be filled with median/mode values and add noise. We drop them.

In [ ]:
cols_to_drop = missing[missing > 0.9].index
df = df.drop(columns=cols_to_drop)

In [ ]:
df.info()

**Data Type Conversion**

Numeric columns like `income` and `loan_amount` were read in as strings because of the `"NA"` / `"Exempt"` values. We force them to float with `errors="coerce"` so unconvertible strings become `NaN` rather than raising an error.

Race, ethnicity, and sex are already strings, but converting them to `category` type reduces memory and tells sklearn's `ColumnTransformer` to treat them as discrete groups during encoding.

In [ ]:
# Convert numeric variables
df["income"] = pd.to_numeric(df["income"], errors="coerce")
df["loan_amount"] = pd.to_numeric(df["loan_amount"], errors="coerce")

# Convert categorical variables
cat_cols = ["derived_race", "derived_ethnicity", "derived_sex"]
for col in cat_cols:
    df[col] = df[col].astype("category")

#### 4.	Baseline fairness analysis (pre-model)

Before fitting any model we check raw approval rates by race, ethnicity, and sex. If gaps exist here they're baked into the data — the model didn't create them. Knowing the baseline lets us later distinguish data-driven disparities from model-amplified ones.

In [ ]:
df.groupby("derived_race")["target"].mean().sort_values()

In [ ]:
import matplotlib.pyplot as plt

df.groupby("derived_race")["target"].mean().sort_values().plot(kind="bar")
plt.ylabel("Approval Rate")
plt.title("Approval Rate by Race")
plt.show()

**Approval Rate by Race**

White and Asian applicants are approved at higher rates than other groups; Black or African American applicants sit at the low end. "Joint" applications (multiple applicants on one file) cluster near the top across all three demographic cuts — likely because joint applicants tend to pool stronger financials. "Free Form Text Only" is lowest, which probably reflects data-quality issues rather than actual lending behavior. These gaps exist before any model is applied.

In [ ]:
df.groupby("derived_ethnicity")["target"].mean()

In [ ]:
df.groupby("derived_ethnicity")["target"].mean().sort_values().plot(kind="bar")
plt.ylabel("Approval Rate")
plt.title("Approval Rate by Ethnicity")
plt.show()

**Approval Rate by Ethnicity**

"Not Hispanic or Latino" and "Joint" are at the top; "Hispanic or Latino" and "Free Form Text Only" are lower. The gap is real but we haven't controlled for financial variables yet, so we can't attribute it to discrimination vs. differences in underlying applications.

In [ ]:
df.groupby("derived_sex")["target"].mean()

In [ ]:
df.groupby("derived_sex")["target"].mean().sort_values().plot(kind="bar")
plt.ylabel("Approval Rate")
plt.title("Approval Rate by Sex")
plt.show()

**Approval Rate by Sex**

Male and female applicants have similar raw approval rates — the gap here is much smaller than what we saw for race. "Joint" is again the highest category. "Sex Not Available" is a data-quality flag, not a demographic finding.

**Examining Control Variables**

Approval gaps by race might just reflect income differences if wealthier applicants cluster in certain racial groups. We check income and DTI by race to see how much those financial variables explain away the gap.

In [ ]:
df.groupby("derived_race").agg({
    "income": "mean",
    "target": "mean"
})

**Comparing Approval Rates and Income Across Racial Groups**

Approval rates range from roughly 63% (Black or African American) to 78–79% (White, Asian). The unexpected finding: Black or African American applicants have the *highest* average income in the dataset, yet the lowest approval rate. Income alone does not close the gap. We still need to control for DTI, loan type, and collateral before drawing any conclusions.

In [ ]:
import numpy as np

def dti_to_numeric(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x)
    
    if "<20%" in x:
        return 15
    elif "20%-<30%" in x:
        return 25
    elif "30%-<36%" in x:
        return 33
    elif "36%-<50%" in x:
        return 43
    elif "50%-60%" in x:
        return 55
    elif ">60%" in x:
        return 65
    
    try:
        return float(x)
    except:
        return np.nan

**Handling and Simplifying Debt-to-Income Ratio (DTI)**

The original `debt_to_income_ratio` variable contains a mixture of formats, including both numeric values (e.g., 40.0, 45.0) and categorical ranges (e.g., "20%-<30%", "50%-60%"). This makes the data inconsistent and difficult to interpret directly, especially when comparing financial risk across groups.

To improve readability and analysis, we aim to group DTI into a small number of meaningful categories (e.g., <30%, 30–40%, 40–50%, etc.) and examine how these risk levels relate to approval rates.

However, in order to classify values using conditional statements (e.g., `if x < 30`), the data must first be in numeric form. Since the original variable includes both numbers and string-based ranges, a direct conversion to numeric would result in many values being converted to `NaN`, causing significant information loss.

To address this, we first map each range-based value to a representative numeric value (e.g., "30%-<36%" → 33). This step allows us to preserve the relative meaning of each range while converting all values into a consistent numeric format.

Once the data is standardized, we can then apply conditional logic to group observations into clearly defined DTI categories, enabling a more structured and interpretable analysis of financial risk and approval outcomes.

In [ ]:
df["dti_numeric"] = df["debt_to_income_ratio"].apply(dti_to_numeric)

In [ ]:
def simplify_dti(x):
    if pd.isna(x):
        return "Missing"
    
    if x < 30:
        return "<30%"
    elif x < 40:
        return "30-40%"
    elif x < 50:
        return "40-50%"
    elif x < 60:
        return "50-60%"
    else:
        return ">60%"

In [ ]:
df["dti_group"] = df["dti_numeric"].apply(simplify_dti)

In [ ]:
result = df.groupby(["derived_race", "dti_group"])["target"].mean().unstack()
result["overall_approval_rate"] = df.groupby("derived_race")["target"].mean()
result

**Comparing Approval Rates and DTI Across Racial Groups**

Within every DTI bucket, approval rates still vary by race. Higher DTI leads to lower approval for everyone, which is expected. But White applicants sit above other groups even at the same debt level. Black or African American and "Two or More Minority Races" applicants tend to have the lowest rates within each DTI category. DTI accounts for part of the gap but not all of it.

### Modeling  

#### 5.	Feature selection

**Model Design and Feature Selection**

To evaluate both predictive performance and fairness, we construct two models with different feature sets.

---

**Model A: Baseline Model (Excluding Protected Attributes)**

Model A includes only financial and loan-related variables that are commonly used in lending decisions:

- **1. income**: Represents the applicant’s repayment ability. Higher income generally indicates lower default risk.

- **2. ebt-to-income ratio (DTI group)**: A key risk indicator reflecting the applicant’s debt burden relative to income.

- **3. loan_term**: Indicates the duration of the loan. Longer terms may be associated with different risk profiles.

- **4. loan_type**: Different loan types (e.g., conventional vs. FHA) have different approval standards and risk characteristics.

- **5. loan_purpose**: Distinguishes between purposes such as home purchase or refinancing, which may affect approval likelihood.

- **6. applicant_credit_score_type**: Indicates the type of credit scoring model used for the applicant. Although it is not the numeric credit score itself, it can still serve as a proxy for credit evaluation differences and may help explain variation in approval patterns or pricing decisions.

- **7. tract_minority_population_percent**: Measures the share of minority residents in the applicant’s census tract. This variable can serve as a neighborhood-level proxy for demographic context and is useful for analyzing potential geographic or structural fairness issues beyond individual protected attributes.

- **8. loan_to_value_ratio**:  Captures the ratio between the loan amount and the property value, directly reflecting the borrower’s leverage level. A higher CLTV indicates that the borrower is financing a larger portion of the property, which typically implies higher credit risk and plays a critical role in underwriting decisions.

- **9. property_value**: Represents the estimated value of the property used as collateral. This feature provides context for the scale of the loan and helps the model assess whether the requested loan amount is reasonable relative to the asset value.


We retain combined **combined_loan-to-value ratio (CLTV)** and **property_value** while excluding **loan_amount** to avoid redundancy. CLTV captures the borrower’s leverage and directly reflects credit risk, while property_value provides contextual information about the scale of the underlying asset. Together, they allow the model to distinguish between relative risk and absolute transaction size, leading to clearer interpretation and more reliable fairness analysis.

We exclude **interest_rate** and **rate_spread** from the main model because they are closely tied to pricing outcomes and may introduce data leakage, as they are not strictly determined before the approval decision. We also exclude **state_code** to avoid capturing broad geographic patterns that may reflect structural or historical disparities rather than individual financial risk. These variables are instead used in post-model analysis, where **interest_rate** and **rate_spread** support pricing fairness evaluation, and **state_code** is used to assess robustness and geographic disparities.

These variables are included because they provide meaningful and justifiable information about financial risk and lending conditions. Model A represents a fairness-aware design that does not directly use protected attributes in decision-making.

---
**Model B: Extended Model (Including Protected Attribute)**

Model B includes all features from Model A, with the addition of:

- **derived_race**: A protected attribute representing the applicant’s race.

This model is not intended for deployment but is used as an analytical tool to examine whether including race improves predictive performance and how it affects fairness. The reason of not adding sex is because we didn't observe the disparity in raw data. 

---

**Purpose of Comparing Two Models**

The goal of constructing these two models is to analyze the trade-off between accuracy and fairness. By comparing Model A and Model B, we can evaluate:

- Whether including race significantly improves predictive accuracy
- Whether it leads to increased disparities across racial groups

In particular, we focus on group-based error metrics such as:

- **False Positive Rate (FPR)**: The rate at which applicants who should be rejected are incorrectly approved
- **False Negative Rate (FNR)**: The rate at which applicants who should be approved are incorrectly rejected

By comparing FPR and FNR across racial groups, we can assess whether the model produces unequal error patterns, which may indicate potential bias.


In [ ]:
from sklearn.model_selection import train_test_split

#features without race 
featuresA = [
    "income",
    "dti_group",
    "loan_term",
    "loan_type",
    "loan_purpose",
    "applicant_credit_score_type",
    "tract_minority_population_percent",
    "loan_to_value_ratio",
    "property_value"
    

]
#features with race 
featuresB= featuresA + ["derived_race"]
target = "target"

#### 6. Train-Test Split

Both models are trained and evaluated on the exact same rows. The only difference is the feature set. Any performance or fairness difference can therefore be attributed to whether race was included, not to sampling variation.

**Model A (without race)**

In [ ]:
# -----------------------------
# Step 0: define feature types
# -----------------------------
numeric_features_A = [
    "income",
    "tract_minority_population_percent",
    "loan_to_value_ratio",
    "property_value"
]

categorical_features_A = [
    "dti_group",
    "loan_term",
    "loan_type",
    "loan_purpose",
    "applicant_credit_score_type"
]

# -----------------------------
# Step 1: clean dtypes in df
# -----------------------------
for col in numeric_features_A:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in categorical_features_A:
    df[col] = df[col].map(lambda x: np.nan if pd.isna(x) else str(x))

# for Model B
df["derived_race"] = df["derived_race"].map(
    lambda x: np.nan if pd.isna(x) else str(x)
)

# -----------------------------
# Step 2: build X and y
# -----------------------------
X_A = df[featuresA].copy()
y = df[target].astype(int).copy()

# -----------------------------
# Step 3: train/test split
# -----------------------------
X_train_A, X_test_A, y_train, y_test = train_test_split(
    X_A,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# optional check
print(X_train_A.dtypes)
print(X_test_A.dtypes)

n_train = X_train_A.shape[0]
n_test = X_test_A.shape[0]
pct_train = 100 * n_train / (n_train + n_test)
pct_test = 100 * n_test / (n_train + n_test)
print(
    f"Model A → Train: {X_train_A.shape} ({n_train:,} rows, {pct_train:.1f}%), "
    f"Test: {X_test_A.shape} ({n_test:,} rows, {pct_test:.1f}%)"
)

**Model B (with race)**

In [ ]:
# Build X
X_B = df[featuresB].copy()

# Use SAME split indices
X_train_B = X_B.loc[X_train_A.index].copy()
X_test_B  = X_B.loc[X_test_A.index].copy()

# Clean dtypes in X_B using same logic as X_A
for col in numeric_features_A:
    X_train_B[col] = pd.to_numeric(X_train_B[col], errors="coerce")
    X_test_B[col]  = pd.to_numeric(X_test_B[col], errors="coerce")

for col in categorical_features_A:
    X_train_B[col] = X_train_B[col].map(lambda x: np.nan if pd.isna(x) else str(x))
    X_test_B[col] = X_test_B[col].map(lambda x: np.nan if pd.isna(x) else str(x))

# Clean derived
X_train_B["derived_race"] = X_train_B["derived_race"].map(
    lambda x: np.nan if pd.isna(x) else str(x)
)
X_test_B["derived_race"] = X_test_B["derived_race"].map(
    lambda x: np.nan if pd.isna(x) else str(x)
)


# optional check
print(X_train_B.dtypes)
print(X_test_B.dtypes)

print(
    f"Model B → Train: {X_train_B.shape}, "
    f"Test: {X_test_B.shape}"
)

#### 7.  logistic regression (Model A vs Model B)

**Model A**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def _cat_value_for_ohe(x):
    import numpy as np
    import pandas as pd

    if x is None:
        return np.nan
    if isinstance(x, (float, np.floating)) and (np.isnan(x) or np.isinf(x)):
        return np.nan
    try:
        if pd.isna(x):
            return np.nan
    except (ValueError, TypeError):
        pass
    return str(x)


def _cat_block_to_str(X):
    import numpy as np
    import pandas as pd

    X = np.asarray(X, dtype=object)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    out = np.empty(X.shape, dtype=object)
    for j in range(X.shape[1]):
        col = pd.Series(np.asarray(X[:, j], dtype=object))
        out[:, j] = col.map(_cat_value_for_ohe).to_numpy(dtype=object, na_value=np.nan)
    return out


# -----------------------------
# Step 4: preprocessing for Model A
# -----------------------------
numeric_transformer_A = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_A = Pipeline(steps=[
    ("as_str_in", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("as_str_out", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_A = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_A, numeric_features_A),
        ("cat", categorical_transformer_A, categorical_features_A)
    ]
)

# -----------------------------
# Step 5: Logistic Regression pipeline
# -----------------------------
model_A = Pipeline(steps=[
    ("preprocessor", preprocessor_A),
    ("classifier", LogisticRegression(max_iter=1000))
])

# -----------------------------
# Step 5.5: guarantee sklearn-safe dtypes (pd.NA → np.nan)
# -----------------------------
for _c in numeric_features_A:
    X_train_A[_c] = pd.to_numeric(X_train_A[_c], errors="coerce").values
    X_test_A[_c]  = pd.to_numeric(X_test_A[_c],  errors="coerce").values
for _c in categorical_features_A:
    X_train_A[_c] = X_train_A[_c].map(lambda x: np.nan if pd.isna(x) else str(x))
    X_test_A[_c] = X_test_A[_c].map(lambda x: np.nan if pd.isna(x) else str(x))

# -----------------------------
# Step 6: fit model
# -----------------------------
model_A.fit(X_train_A, y_train)

# -----------------------------
# Step 7: predictions
# -----------------------------
y_train_pred_A = model_A.predict(X_train_A)
y_test_pred_A = model_A.predict(X_test_A)

# -----------------------------
# Step 8: evaluation
# -----------------------------
train_acc_A = accuracy_score(y_train, y_train_pred_A)
test_acc_A = accuracy_score(y_test, y_test_pred_A)

print("========== Model A: Logistic Regression ==========")
print(f"Train Accuracy: {train_acc_A:.4f}")
print(f"Test Accuracy:  {test_acc_A:.4f}")

In [ ]:
result_A = X_test_A.copy()
result_A["y_true"] = y_test.values
result_A["y_pred"] = y_test_pred_A
result_A["race"] = df.loc[X_test_A.index, "derived_race"].astype(object).fillna(np.nan)

def compute_metrics(group):
    tn, fp, fn, tp = confusion_matrix(
        group["y_true"],
        group["y_pred"],
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    acc = (tp + tn) / (tp + tn + fp + fn)
    approval_rate = group["y_pred"].mean()

    return pd.Series({
        "Count": len(group),
        "Approval_Rate": approval_rate,
        "FPR": fpr,
        "FNR": fnr,
        "Accuracy": acc
    })

fairness_A = result_A.groupby("race", dropna=False).apply(compute_metrics)
fairness_A.round(4)

**Model B**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def _cat_value_for_ohe(x):
    import numpy as np
    import pandas as pd

    if x is None:
        return np.nan
    if isinstance(x, (float, np.floating)) and (np.isnan(x) or np.isinf(x)):
        return np.nan
    try:
        if pd.isna(x):
            return np.nan
    except (ValueError, TypeError):
        pass
    return str(x)


def _cat_block_to_str(X):
    import numpy as np
    import pandas as pd

    X = np.asarray(X, dtype=object)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    out = np.empty(X.shape, dtype=object)
    for j in range(X.shape[1]):
        col = pd.Series(np.asarray(X[:, j], dtype=object))
        out[:, j] = col.map(_cat_value_for_ohe).to_numpy(dtype=object, na_value=np.nan)
    return out


# -----------------------------
# Step 1: build X for Model B
# -----------------------------
X_B = df[featuresB].copy()

# Use the same split indices as Model A
X_train_B = X_B.loc[X_train_A.index].copy()
X_test_B  = X_B.loc[X_test_A.index].copy()

# -----------------------------
# Step 2: define feature types for Model B
# -----------------------------
numeric_features_B = numeric_features_A.copy()

categorical_features_B = categorical_features_A + ["derived_race"]

# guarantee sklearn-safe dtypes (pd.NA → np.nan)
for col in numeric_features_B:
    X_train_B[col] = pd.to_numeric(X_train_B[col], errors="coerce").values
    X_test_B[col]  = pd.to_numeric(X_test_B[col],  errors="coerce").values
for col in categorical_features_B:
    X_train_B[col] = X_train_B[col].map(lambda x: np.nan if pd.isna(x) else str(x))
    X_test_B[col] = X_test_B[col].map(lambda x: np.nan if pd.isna(x) else str(x))

# -----------------------------
# Step 3: preprocessing for Model B
# -----------------------------
numeric_transformer_B = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_B = Pipeline(steps=[
    ("as_str_in", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("as_str_out", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_B = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_B, numeric_features_B),
        ("cat", categorical_transformer_B, categorical_features_B)
    ]
)

# -----------------------------
# Step 4: Logistic Regression pipeline
# -----------------------------
model_B = Pipeline(steps=[
    ("preprocessor", preprocessor_B),
    ("classifier", LogisticRegression(max_iter=1000))
])

# -----------------------------
# Step 5: fit Model B
# -----------------------------
model_B.fit(X_train_B, y_train)

# -----------------------------
# Step 6: predictions
# -----------------------------
y_train_pred_B = model_B.predict(X_train_B)
y_test_pred_B = model_B.predict(X_test_B)

# -----------------------------
# Step 7: evaluation
# -----------------------------
train_acc_B = accuracy_score(y_train, y_train_pred_B)
test_acc_B = accuracy_score(y_test, y_test_pred_B)

print("========== Model B: Logistic Regression (+ race) ==========")
print(f"Train Accuracy: {train_acc_B:.4f}")
print(f"Test Accuracy:  {test_acc_B:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd

# ---------- Predict ----------
pred_B = model_B.predict(X_test_B)

# Combine results
result_B = X_test_B.copy()
result_B["y_true"] = y_test.values
result_B["y_pred"] = pred_B
result_B["race"] = df.loc[X_test_B.index, "derived_race"].astype(object).fillna(np.nan)

# ---------- Define metrics ----------
def compute_metrics(group):
    tn, fp, fn, tp = confusion_matrix(
        group["y_true"],
        group["y_pred"],
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    acc = (tp + tn) / (tp + tn + fp + fn)
    approval_rate = group["y_pred"].mean()

    return pd.Series({
        "Count": len(group),
        "Approval_Rate": approval_rate,
        "FPR": fpr,
        "FNR": fnr,
        "Accuracy": acc
    })

# ---------- Group by race ----------
fairness_B = result_B.groupby("race", dropna=False).apply(compute_metrics)

fairness_B.round(4)

**Comparison of Accuracy, FNR, FPR between Model A (without race)and Model B (race)**

In [ ]:
fairness_A_renamed = fairness_A.rename(columns={
    "FPR": "FPR_A",
    "FNR": "FNR_A",
    "Accuracy": "Accuracy_A",
    "Approval_Rate": "Approval_Rate_A"
}).drop(columns=["Count"])

fairness_B_renamed = fairness_B.rename(columns={
    "FPR": "FPR_B",
    "FNR": "FNR_B",
    "Accuracy": "Accuracy_B",
    "Approval_Rate": "Approval_Rate_B"
}).drop(columns=["Count"])

comparison_lr = fairness_A_renamed.join(fairness_B_renamed)
comparison_lr.round(4)

In [ ]:
print("========== Model A: Logistic Regression ==========")
print(f"Train Accuracy: {train_acc_A:.4f}")
print(f"Test Accuracy:  {test_acc_A:.4f}")
print("========== Model B: Logistic Regression (+ race) ==========")
print(f"Train Accuracy: {train_acc_B:.4f}")
print(f"Test Accuracy:  {test_acc_B:.4f}")

**Interpretation of Results**

Model B (with race) barely moves overall accuracy compared to Model A. But for Black or African American and American Indian or Alaska Native applicants, approval rates drop and FNR rises — meaning more qualified applicants from those groups get incorrectly denied. Including race explicitly did not help fairness; in practice it made outcomes worse for the groups it was meant to capture.

#### 8. Gradient boost tree (Model A vs Model B)

**Model A**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -----------------------------
# Step: Gradient Boosting pipeline (Model A)
# -----------------------------
model_A_gbt = Pipeline(steps=[
    ("preprocessor", preprocessor_A),
    ("classifier", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

# -----------------------------
# Fit model
# -----------------------------
model_A_gbt.fit(X_train_A, y_train)

# -----------------------------
# Predictions
# -----------------------------
y_train_pred_gbt = model_A_gbt.predict(X_train_A)
y_test_pred_gbt = model_A_gbt.predict(X_test_A)

# -----------------------------
# Accuracy
# -----------------------------
train_acc_gbt = accuracy_score(y_train, y_train_pred_gbt)
test_acc_gbt = accuracy_score(y_test, y_test_pred_gbt)

print("========== Model A: Gradient Boosting ==========")
print(f"Train Accuracy: {train_acc_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_gbt:.4f}")


### 8.1 Test ROC-AUC — same fitted models (no retraining)

Logistic regression (`model_A`) and gradient boosting (`model_A_gbt`) are already fit above. This cell only reports **test-set ROC-AUC** from their existing `predict_proba` outputs so the written summary (e.g. LR ≈ 0.805 vs GBT ≈ 0.847) matches a single visible line in the notebook.

In [ ]:
from sklearn.metrics import roc_auc_score

y_test_prob_lr = model_A.predict_proba(X_test_A)[:, 1]
y_test_prob_gbt = model_A_gbt.predict_proba(X_test_A)[:, 1]
test_auc_lr = roc_auc_score(y_test, y_test_prob_lr)
test_auc_gbt = roc_auc_score(y_test, y_test_prob_gbt)

print("========== Test ROC-AUC (Model A, already trained) ==========")
print(f"  Logistic Regression — test AUC: {test_auc_lr:.4f}")
print(f"  Gradient Boosting   — test AUC: {test_auc_gbt:.4f}")

### 8.2 Ablation: without `tract_minority_population_percent`

We **retrain** separate pipelines that drop this one numeric feature (same split, same hyperparameters as main Model A). Comparing **test AUC** and **predicted approval AIR (Black / White)** shows the accuracy–fairness tradeoff of keeping a neighborhood proxy.

*Note: full GBT refit on ~6.9M rows can take several minutes.*

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd


def _cat_value_for_ohe(x):
    import numpy as np
    import pandas as pd

    if x is None:
        return np.nan
    if isinstance(x, (float, np.floating)) and (np.isnan(x) or np.isinf(x)):
        return np.nan
    try:
        if pd.isna(x):
            return np.nan
    except (ValueError, TypeError):
        pass
    return str(x)


def _cat_block_to_str(X):
    import numpy as np
    import pandas as pd

    X = np.asarray(X, dtype=object)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    out = np.empty(X.shape, dtype=object)
    for j in range(X.shape[1]):
        col = pd.Series(np.asarray(X[:, j], dtype=object))
        out[:, j] = col.map(_cat_value_for_ohe).to_numpy(dtype=object, na_value=np.nan)
    return out


TR = "tract_minority_population_percent"
numeric_no = [c for c in numeric_features_A if c != TR]

_num = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
_cat = Pipeline(steps=[
    ("as_str_in", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("as_str_out", FunctionTransformer(_cat_block_to_str, validate=False)),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
pre_no = ColumnTransformer(
    transformers=[
        ("num", _num, numeric_no),
        ("cat", _cat, categorical_features_A),
    ]
)

Xtr = X_train_A.drop(columns=[TR])
Xte = X_test_A.drop(columns=[TR])
for _c in numeric_no:
    Xtr[_c] = pd.to_numeric(Xtr[_c], errors="coerce").values
    Xte[_c] = pd.to_numeric(Xte[_c], errors="coerce").values
for _c in categorical_features_A:
    Xtr[_c] = Xtr[_c].map(lambda x: np.nan if pd.isna(x) else str(x))
    Xte[_c] = Xte[_c].map(lambda x: np.nan if pd.isna(x) else str(x))

model_lr_no = Pipeline([
    ("preprocessor", pre_no),
    ("classifier", LogisticRegression(max_iter=1000)),
])
model_gbt_no = Pipeline([
    ("preprocessor", pre_no),
    ("classifier", GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
    )),
])

model_lr_no.fit(Xtr, y_train)
model_gbt_no.fit(Xtr, y_train)

race_test = df.loc[X_test_A.index, "derived_race"]

def air_bw(y_pred, race_ser):
    b = (race_ser == "Black or African American").to_numpy()
    w = (race_ser == "White").to_numpy()
    if b.sum() == 0 or w.sum() == 0:
        return float("nan")
    return float(y_pred[b].mean() / y_pred[w].mean())

auc_lr_no = roc_auc_score(y_test, model_lr_no.predict_proba(Xte)[:, 1])
auc_gbt_no = roc_auc_score(y_test, model_gbt_no.predict_proba(Xte)[:, 1])
air_lr_no = air_bw(model_lr_no.predict(Xte), race_test)
air_gbt_no = air_bw(model_gbt_no.predict(Xte), race_test)

print("========== Ablation: TRACT feature removed (retrained) ==========")
print(f"  LR  — test AUC: {auc_lr_no:.4f}  |  pred. AIR Black/White: {air_lr_no:.4f}")
print(f"  GBT — test AUC: {auc_gbt_no:.4f}  |  pred. AIR Black/White: {air_gbt_no:.4f}")
print("========== Baseline: WITH tract (same test set) ==========")
print(f"  LR  — test AUC: {test_auc_lr:.4f}  |  pred. AIR Black/White: {air_bw(model_A.predict(X_test_A), race_test):.4f}")
print(f"  GBT — test AUC: {test_auc_gbt:.4f}  |  pred. AIR Black/White: {air_bw(model_A_gbt.predict(X_test_A), race_test):.4f}")

### 8.3 USDA (`loan_type` = 4): dedicated GBT on USDA-only training rows + human-review flag

HMDA uses **4** for USDA rural housing loans. The slice analysis above shows poor performance on this segment. Here we (1) train a **separate** GBT on **training** rows with `loan_type == 4` only, and report AUC on the **USDA test** rows; (2) add a simple **`needs_human_review` flag** (USDA) as an operational mitigation.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
import numpy as np
import pandas as pd


def _cat_value_for_ohe(x):
    import numpy as np
    import pandas as pd

    if x is None:
        return np.nan
    if isinstance(x, (float, np.floating)) and (np.isnan(x) or np.isinf(x)):
        return np.nan
    try:
        if pd.isna(x):
            return np.nan
    except (ValueError, TypeError):
        pass
    return str(x)


def _cat_block_to_str(X):
    import numpy as np
    import pandas as pd

    X = np.asarray(X, dtype=object)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    out = np.empty(X.shape, dtype=object)
    for j in range(X.shape[1]):
        col = pd.Series(np.asarray(X[:, j], dtype=object))
        out[:, j] = col.map(_cat_value_for_ohe).to_numpy(dtype=object, na_value=np.nan)
    return out


lt_train = pd.to_numeric(X_train_A["loan_type"], errors="coerce")
lt_test = pd.to_numeric(X_test_A["loan_type"], errors="coerce")
mask_tr = lt_train == 4
mask_te = lt_test == 4

n_tr, n_te = int(mask_tr.sum()), int(mask_te.sum())
print("========== USDA segment (loan_type == 4) ==========")
print(f"  Train count: {n_tr:,}  |  Test count: {n_te:,}")

# Human-in-the-loop flag on the test frame (example policy)
needs_human_review = mask_te.to_numpy()
print(f"  Test rows flagged for human review (USDA): {needs_human_review.sum():,}")

if n_tr < 500 or n_te < 50:
    print("  [Skip sub-model] Not enough USDA rows to fit a stable GBT on this machine run.")
else:
    X_us_tr = X_train_A.loc[mask_tr]
    y_us_tr = y_train.loc[mask_tr]
    X_us_te = X_test_A.loc[mask_te]
    y_us_te = y_test.loc[mask_te]

    for _c in numeric_features_A:
        X_us_tr[_c] = pd.to_numeric(X_us_tr[_c], errors="coerce").values
        X_us_te[_c] = pd.to_numeric(X_us_te[_c], errors="coerce").values
    for _c in categorical_features_A:
        X_us_tr[_c] = X_us_tr[_c].map(lambda x: np.nan if pd.isna(x) else str(x))
        X_us_te[_c] = X_us_te[_c].map(lambda x: np.nan if pd.isna(x) else str(x))

    _num_u = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    _cat_u = Pipeline(steps=[
        ("as_str_in", FunctionTransformer(_cat_block_to_str, validate=False)),
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("as_str_out", FunctionTransformer(_cat_block_to_str, validate=False)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    pre_us = ColumnTransformer(
        transformers=[
            ("num", _num_u, numeric_features_A),
            ("cat", _cat_u, categorical_features_A),
        ]
    )
    model_usda_gbt = Pipeline([
        ("preprocessor", pre_us),
        ("classifier", GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
        )),
    ])
    model_usda_gbt.fit(X_us_tr, y_us_tr)

    p_global = model_A_gbt.predict_proba(X_us_te)[:, 1]
    p_sub = model_usda_gbt.predict_proba(X_us_te)[:, 1]
    if y_us_te.nunique() < 2:
        print("  [Skip AUC] USDA test slice has only one class in `target`.")
    else:
        auc_global = roc_auc_score(y_us_te, p_global)
        auc_sub = roc_auc_score(y_us_te, p_sub)
        print(f"  GBT (global model) AUC on USDA test: {auc_global:.4f}")
        print(f"  GBT (USDA-only train) AUC on USDA test: {auc_sub:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd

# ---------- Predict ----------
pred_gbt = model_A_gbt.predict(X_test_A)

# ---------- Combine results ----------
result_gbt = X_test_A.copy()
result_gbt["y_true"] = y_test.values
result_gbt["y_pred"] = pred_gbt

result_gbt["race"] = df.loc[X_test_A.index, "derived_race"].astype(object).fillna(np.nan)

# ---------- Define metrics ----------
def compute_metrics(group):
    tn, fp, fn, tp = confusion_matrix(
        group["y_true"],
        group["y_pred"],
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    acc = (tp + tn) / (tp + tn + fp + fn)
    approval_rate = group["y_pred"].mean()

    return pd.Series({
        "Count": len(group),
        "Approval_Rate": approval_rate,
        "FPR": fpr,
        "FNR": fnr,
        "Accuracy": acc
    })

# ---------- Group by race ----------
fairness_gbt = result_gbt.groupby("race", dropna=False).apply(compute_metrics)
fairness_gbt.round(4)

**Model B**

In [ ]:
# -----------------------------
# Gradient Boosting pipeline
# -----------------------------
model_B_gbt = Pipeline(steps=[
    ("preprocessor", preprocessor_B),
    ("classifier", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

# -----------------------------
# fit model
# -----------------------------
model_B_gbt.fit(X_train_B, y_train)

# -----------------------------
# predictions
# -----------------------------
y_train_pred_B_gbt = model_B_gbt.predict(X_train_B)
y_test_pred_B_gbt = model_B_gbt.predict(X_test_B)

# -----------------------------
# evaluation
# -----------------------------
train_acc_B_gbt = accuracy_score(y_train, y_train_pred_B_gbt)
test_acc_B_gbt = accuracy_score(y_test, y_test_pred_B_gbt)

print("========== Model B: Gradient Boosting (+ race) ==========")
print(f"Train Accuracy: {train_acc_B_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_B_gbt:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd

# ---------- Predict ----------
pred_B_gbt = model_B_gbt.predict(X_test_B)

# ---------- Combine results ----------
result_B_gbt = X_test_B.copy()
result_B_gbt["y_true"] = y_test.values
result_B_gbt["y_pred"] = pred_B_gbt
result_B_gbt["race"] = df.loc[X_test_B.index, "derived_race"].astype(object).fillna(np.nan)

# ---------- Define metrics ----------
def compute_metrics(group):
    tn, fp, fn, tp = confusion_matrix(
        group["y_true"],
        group["y_pred"],
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    acc = (tp + tn) / (tp + tn + fp + fn)
    approval_rate = group["y_pred"].mean()

    return pd.Series({
        "Approval_Rate": approval_rate,
        "FPR": fpr,
        "FNR": fnr,
        "Accuracy": acc
    })

# ---------- Group by race ----------
fairness_B_gbt = result_B_gbt.groupby("race", dropna=False).apply(compute_metrics)

fairness_B_gbt.round(4)

**Comparison of Accuracy, FNR, FPR between Model A (without race)and Model B (race)**

In [ ]:
fairness_gbt_A_renamed = fairness_gbt.rename(columns={
    "FPR": "FPR_A_GBT",
    "FNR": "FNR_A_GBT",
    "Accuracy": "Accuracy_A_GBT",
    "Approval_Rate": "Approval_Rate_A_GBT"
}).drop(columns=["Count"], errors="ignore")

fairness_gbt_B_renamed = fairness_B_gbt.rename(columns={
    "FPR": "FPR_B_GBT",
    "FNR": "FNR_B_GBT",
    "Accuracy": "Accuracy_B_GBT",
    "Approval_Rate": "Approval_Rate_B_GBT"
}).drop(columns=["Count"], errors="ignore")

comparison_gbt = fairness_gbt_A_renamed.join(fairness_gbt_B_renamed)
comparison_gbt.round(4)

In [ ]:
print("========== Model A: Gradient Boosting ==========")
print(f"Train Accuracy: {train_acc_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_gbt:.4f}")
print("========== Model B: Gradient Boosting (+ race) ==========")
print(f"Train Accuracy: {train_acc_B_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_B_gbt:.4f}")

**Interpretation of GBT Results**

GBT beats logistic regression on accuracy and error rates across the board. The HMDA data has nonlinear interactions between income, DTI, and loan type that a linear model cannot capture. The performance gap is consistent, so we use GBT (Model A) as the primary model for robustness and security analysis.

#### 9. Comparison Across Modeling Approaches

**Why We Use Multiple Models?**

We run both LR and GBT for two reasons: (1) interpretability — LR gives readable coefficients and clean gradient signals needed for the PGD attack; (2) consistency check — if both models produce the same fairness findings, the result reflects real data patterns rather than model-specific behavior.

**Performance from Different Approaches**

To summarize the results across different modeling approaches, we present the comparison tables below.

In [ ]:
print("Logistic Regression")
display(comparison_lr)

print("Gradient Boosted Tree")
display(comparison_gbt)

In [ ]:
print("========== Model A: Logistic Regression ==========")
print(f"Train Accuracy: {train_acc_A:.4f}")
print(f"Test Accuracy:  {test_acc_A:.4f}")
print("========== Model B: Logistic Regression (+ race) ==========")
print(f"Train Accuracy: {train_acc_B:.4f}")
print(f"Test Accuracy:  {test_acc_B:.4f}")


print("========== Model A: Gradient Boosting ==========")
print(f"Train Accuracy: {train_acc_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_gbt:.4f}")
print("========== Model B: Gradient Boosting (+ race) ==========")
print(f"Train Accuracy: {train_acc_B_gbt:.4f}")
print(f"Test Accuracy:  {test_acc_B_gbt:.4f}")

GBT is consistently more accurate than LR, but racial disparities in approval rates, FPR, and FNR persist. Including race (Model B) shifts some numbers but does not fix the underlying gap. The disparities survive model choice, which points to proxy variables in the data — particularly tract-level features correlated with race — rather than a modeling artifact.

#### 10. Interpretation and Insights

**Interpretation of Remaining Disparities**

Income explained part of the approval gap; DTI explained more. But neither closes it. When race is added as a feature (Model B), error rates shift in the wrong direction for minority groups. The residual pattern points to proxy variables like tract demographics that carry racial information even when race itself is excluded from the model.

**Modeling Strategy and Key Insights**

Model A (no race) is our primary analysis model — it reflects what a lender would actually deploy under ECOA/FHA constraints. Model B (with race) is a diagnostic tool: if adding race makes outcomes worse for minority groups, that reveals how proxy information travels through other features.

Both models go forward into interpretability, fairness, and robustness analysis, but the main findings and recommendations are anchored to Model A.

### Transparency (Model Explainability)

Transparency

These features help explain why the model makes certain decisions and reflect core financial logic.

**1. income**: Represents the applicant’s repayment ability. Higher income generally indicates lower default risk.

**2. debt-to-income ratio (DTI group)**: A key risk indicator reflecting the applicant’s debt burden relative to income.

**3. combined loan-to-value ratio (CLTV)**: Measures leverage relative to property value, indicating financial risk exposure.

**4. loan_amount**: Captures the size of the loan. Larger loans may carry higher financial risk.

**5. loan_term:** Indicates the duration of the loan. Longer terms may affect repayment stability and risk exposure.

👉 These variables are typically highlighted in:
•	SHAP / LIME explanations
•	Feature importance analysis
•	Decision interpretation

### Fairness Evaluation

Fairness

These features are used to evaluate whether model outcomes differ systematically across groups and to identify potential sources of bias.

1. derived_race / derived_ethnicity / derived_sex: Protected attributes used to define groups for fairness evaluation (e.g., approval rate, FPR, FNR comparisons).
2. tract_minority_population_percent: A neighborhood-level proxy that may capture structural or geographic disparities correlated with race.
3. income: Helps assess whether differences in outcomes can be explained by repayment ability.
4. debt-to-income ratio (DTI group): A key risk factor used to evaluate whether disparities persist after controlling for financial risk.
5. combined loan-to-value ratio (CLTV): Captures leverage and financial exposure, allowing us to test whether risk-based factors explain group differences.
6. loan_type / loan_purpose: Control variables that help distinguish policy or product-related differences from potential bias.

👉 These variables are typically used in:  
• Group fairness metrics (Approval Rate, FPR, FNR)  
• Disparity decomposition (what explains vs what remains)  
• Intersectional analysis (e.g., race × income)

### Lecture 4 Robustness Analysis

Robustness

We check whether performance and fairness hold up across conditions the model was not explicitly trained on: different loan types, geographic regions, economic stress scenarios, and demographic slices.

- `loan_type` / `loan_purpose`: different product segments may have different risk profiles
- `income` / DTI / CLTV: define risk-based slices (low vs. high risk borrowers)
- `tract_minority_population_percent`: tests whether model behavior is stable across neighborhood composition
- `state_code`: captures geographic variation

#### 4.1 Distribution Shift Detection: PSI and KS Test

**Population Stability Index (PSI)** measures how much a feature's distribution has shifted between the train and test sets.
- PSI < 0.10: **Stable** — no action needed
- 0.10 ≤ PSI < 0.25: **Monitor** — investigate the shift
- PSI ≥ 0.25: **Retrain** — distribution has significantly changed

**Kolmogorov-Smirnov (KS) Test** provides a non-parametric statistical test for whether two samples come from the same distribution. A small p-value (< 0.05) indicates a statistically significant difference between train and test distributions.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Subsample train and test for computational efficiency (full dataset is ~8M rows)
SAMPLE_SIZE = 50_000
rng = np.random.default_rng(42)
train_idx = rng.choice(len(X_train_A), size=min(SAMPLE_SIZE, len(X_train_A)), replace=False)
test_idx  = rng.choice(len(X_test_A),  size=min(SAMPLE_SIZE, len(X_test_A)),  replace=False)

X_train_sample = X_train_A.iloc[train_idx]
X_test_sample  = X_test_A.iloc[test_idx]


def compute_psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    """Compute Population Stability Index using percentile-based bin boundaries."""
    breakpoints = np.unique(np.percentile(expected, np.linspace(0, 100, bins + 1)))
    exp_pcts = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    act_pcts = np.histogram(actual,   bins=breakpoints)[0] / len(actual)
    # Replace zeros to avoid log(0) / division by zero
    exp_pcts = np.where(exp_pcts == 0, 1e-6, exp_pcts)
    act_pcts = np.where(act_pcts == 0, 1e-6, act_pcts)
    return float(np.sum((act_pcts - exp_pcts) * np.log(act_pcts / exp_pcts)))


psi_ks_rows = []
for feat in numeric_features_A:
    train_vals = X_train_sample[feat].dropna().values.astype(float)
    test_vals  = X_test_sample[feat].dropna().values.astype(float)

    psi_val            = compute_psi(train_vals, test_vals)
    ks_stat, ks_pval   = stats.ks_2samp(train_vals, test_vals)

    if psi_val < 0.10:
        flag = "STABLE"
    elif psi_val < 0.25:
        flag = "MONITOR"
    else:
        flag = "RETRAIN"

    psi_ks_rows.append({
        "Feature":      feat,
        "PSI":          round(psi_val, 4),
        "PSI_Flag":     flag,
        "KS_Statistic": round(ks_stat, 4),
        "KS_p_value":   round(ks_pval, 6),
    })

psi_df = pd.DataFrame(psi_ks_rows)
print("Distribution Shift Detection — PSI and KS Test (Train vs Test)")
display(psi_df)

**PSI / KS Interpretation**

All four numeric features show PSI < 0.10 (STABLE) and KS p-values well above 0.05.
The train–test split reflects a random 80/20 split of the same 2024 HMDA snapshot,
so stable results are expected. The real production risk is *future* drift (2025+ data)
when interest rates, applicant demographics, or product mix may shift materially.
A PSI monitoring pipeline should be deployed before production.

> **Action**: All features are STABLE — no immediate retraining required. Schedule automated monthly PSI checks before any production release.

#### 4.2 Multivariate Shift: Maximum Mean Discrepancy (MMD)

**MMD²** tests whether the *joint* feature distribution (in the encoded feature space) has shifted between train and test. Unlike PSI/KS which test each feature independently, MMD captures correlations across all features simultaneously.

Thresholds (unbiased linear kernel estimator):
- MMD² < 0.01: **Low shift** — distributions are similar, model is stable
- 0.01 ≤ MMD² < 0.10: **Moderate shift** — monitor model performance
- MMD² ≥ 0.10: **High shift** — significant multivariate drift, consider retraining

In [ ]:
import scipy.sparse as sp

# Subsample to 10 000 rows each side for MMD (quadratic cost in sample size)
MMD_SIZE  = 10_000
rng_mmd   = np.random.default_rng(42)
tr_mmd_idx = rng_mmd.choice(len(X_train_A), size=min(MMD_SIZE, len(X_train_A)), replace=False)
te_mmd_idx = rng_mmd.choice(len(X_test_A),  size=min(MMD_SIZE, len(X_test_A)),  replace=False)

# Transform through the fitted preprocessor (already fitted during modeling)
X_tr_enc = preprocessor_A.transform(X_train_A.iloc[tr_mmd_idx])
X_te_enc = preprocessor_A.transform(X_test_A.iloc[te_mmd_idx])

# Convert sparse matrices to dense arrays if needed
if sp.issparse(X_tr_enc):
    X_tr_enc = X_tr_enc.toarray()
if sp.issparse(X_te_enc):
    X_te_enc = X_te_enc.toarray()

X_tr_enc = X_tr_enc.astype(np.float64)
X_te_enc = X_te_enc.astype(np.float64)


def compute_mmd_linear(X: np.ndarray, Y: np.ndarray) -> float:
    """Unbiased MMD² estimator using linear (dot-product) kernel."""
    n, m = len(X), len(Y)
    XX = np.dot(X, X.T)
    YY = np.dot(Y, Y.T)
    XY = np.dot(X, Y.T)
    mmd2 = (
        (np.sum(XX) - np.trace(XX)) / (n * (n - 1))
        - 2.0 * np.sum(XY) / (n * m)
        + (np.sum(YY) - np.trace(YY)) / (m * (m - 1))
    )
    return float(mmd2)


mmd2 = compute_mmd_linear(X_tr_enc, X_te_enc)
print(f"MMD² (linear kernel, encoded feature space): {mmd2:.6f}")

if mmd2 < 0.01:
    print("Interpretation: LOW shift — train and test distributions are similar; model is stable.")
elif mmd2 < 0.10:
    print("Interpretation: MODERATE shift — some multivariate drift detected; monitor model performance closely.")
else:
    print("Interpretation: HIGH shift — significant distribution change detected; consider retraining.")

**MMD² Interpretation**

The unbiased linear-kernel MMD² = **0.000264** falls well below the Low-shift threshold of 0.01.
This confirms that the *joint* multivariate feature distribution is essentially identical between
the train and test splits — consistent with the per-feature STABLE flags from PSI/KS.

Because both the marginal (PSI/KS) and joint (MMD²) diagnostics agree, we have strong evidence
that the 80/20 split introduced no systematic distributional bias. The model’s generalisation
performance on the test set is therefore a reliable estimate of held-out performance.

> **Caveat**: Both diagnostics compare *train vs. test* drawn from the same 2024 HMDA vintage.
> Production drift from 2025+ economic changes (rate cycles, demographic shifts) is a separate,
> unquantified risk that these tests cannot detect.

#### 4.3 Slice-Based Evaluation (GBT Model A)

We measure model performance (Accuracy, AUC, FPR, FNR) on sub-populations defined by `derived_race` and `loan_type`.

- **AUC Drop** = Overall AUC − Slice AUC. A drop > 0.05 is flagged as potentially problematic.
- **FPR Flag** = FPR > 0.60 triggers a review flag (high rate of false approvals relative to true negatives).

Slice-based evaluation is critical because aggregate metrics can mask serious performance disparities for specific demographic groups — a phenomenon known as Simpson's Paradox.

### ⚠ FLAG Interpretation: loan_type 4 (USDA Loans) — Critical Performance Failure

**Finding:** The USDA loan slice (n = 8,717) recorded AUC = **0.6271** and FPR = **0.9016**, flagged by our AUC-drop threshold (−0.2199, or 22 percentage points below the overall AUC of 0.8470). This is the most severe performance flag in this project.

**What FPR = 0.9016 means:** The model incorrectly approves over 90% of applicants who should actually be denied in this segment — a significant credit risk concern that would attract immediate regulatory scrutiny from OCC and CFPB, independent of any fairness analysis.

---

**Why does this happen? (Three root-cause hypotheses)**

| # | Hypothesis | Evidence | How to Verify |
|---|-----------|----------|---------------|
| 1 | **Insufficient training data** | n = 8,717 is only ~0.5% of the FHA slice (221,422). GBT decision trees need enough examples per subgroup to learn stable decision boundaries. | Check the count of loan_type = 4 records in the training set. |
| 2 | **Feature distribution mismatch** | USDA loans are government-guaranteed products targeting rural, low-to-moderate income borrowers. The distributions of `income` and `loan_to_value_ratio` — the model's most important features — are structurally different from conventional loans. | Compare median `income` for loan_type = 4 vs. loan_type = 1. |
| 3 | **Target label misalignment** | USDA guarantee approvals involve a two-step process (lender + USDA agency review) that may not map cleanly to the `action_taken` binary label used to define our target variable. | Review the target construction logic (Cells 8–11) filtered to loan_type = 4. |

---

**Operational implication:**

> **Model A should NOT be deployed for USDA loan applications in its current form.**
> This subgroup requires either a dedicated sub-model trained specifically on USDA loan data, or a rule-based human-review escalation path for all loan_type = 4 applications.

**Recommended addition to the Residual Risk Register (Q4):**

| Risk | Likelihood | Impact | Mitigation | Status |
|------|-----------|--------|-----------|--------|
| USDA loan slice failure (AUC = 0.6271, FPR = 0.9016) | High | High | Dedicated USDA sub-model or mandatory human review escalation | Open |

---

**Note on other FLAGs:** loan_type = 2 (FHA, AUC = 0.7889, −0.0581) and loan_type = 3 (VA, AUC = 0.7965, −0.0505) are also flagged but at a less critical level (~5–6pp AUC drop). These may reflect the same structural feature-distribution mismatch for government-backed products but do not approach random-chance performance. Monitor with retraining if AUC drop exceeds 10pp in production.

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

# Align derived_race from the original dataframe using shared index
test_race_series = df.loc[X_test_A.index, "derived_race"]

# GBT Model A predictions on the full test set
y_test_prob_gbt = model_A_gbt.predict_proba(X_test_A)[:, 1]
y_test_pred_gbt = model_A_gbt.predict(X_test_A)
y_test_arr      = y_test.values


def compute_slice_metrics(y_true, y_pred, y_prob, label):
    """Return a dict of performance metrics for a given data slice."""
    if len(y_true) < 100:
        return None
    acc = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float("nan")
    return {
        "Slice":    label,
        "n":        len(y_true),
        "Accuracy": round(acc, 4),
        "AUC":      round(auc, 4),
        "FPR":      round(fpr, 4),
        "FNR":      round(fnr, 4),
    }


# Overall baseline row
overall_row = compute_slice_metrics(y_test_arr, y_test_pred_gbt, y_test_prob_gbt, "OVERALL")
overall_auc = overall_row["AUC"]
slice_rows  = [overall_row]

# Slice by derived_race
race_arr = test_race_series.values
for race_val in sorted(test_race_series.dropna().unique()):
    mask = race_arr == race_val
    row  = compute_slice_metrics(
        y_test_arr[mask], y_test_pred_gbt[mask], y_test_prob_gbt[mask],
        f"Race: {race_val}"
    )
    if row:
        slice_rows.append(row)

# Slice by loan_type
loan_type_arr = X_test_A["loan_type"].values
for lt_val in sorted(X_test_A["loan_type"].dropna().unique()):
    mask = loan_type_arr == lt_val
    row  = compute_slice_metrics(
        y_test_arr[mask], y_test_pred_gbt[mask], y_test_prob_gbt[mask],
        f"loan_type: {lt_val}"
    )
    if row:
        slice_rows.append(row)

# Assemble and flag results
slice_df = pd.DataFrame(slice_rows)
slice_df["AUC_Drop"] = (overall_auc - slice_df["AUC"]).round(4)
slice_df["FLAG"]     = slice_df.apply(
    lambda r: "FLAG" if (r["AUC_Drop"] > 0.05 or r["FPR"] > 0.60) else "", axis=1
)

print("Slice-Based Evaluation — GBT Model A")
display(slice_df.sort_values("AUC_Drop", ascending=False).reset_index(drop=True))

#### 4.4 Stress Testing: Sensitivity Under Economic Shocks

We stress-test the model by applying realistic economic shocks to key numeric features and measuring the resulting change in approval rates by race group.

- **Scenario 1**: Income drops 20% (`income × 0.8`) — simulates a recession or job loss event
- **Scenario 2**: Combined Loan-to-Value rises 20% (`loan_to_value_ratio × 1.2`) — simulates falling property values

A robust and fair model should exhibit proportionally consistent changes across demographic groups. Disproportionate impacts on minority groups would indicate that the model amplifies economic shocks for already-disadvantaged populations.

In [ ]:
import matplotlib.pyplot as plt

# Subsample the test set for stress testing
STRESS_SIZE = 50_000
rng_stress  = np.random.default_rng(42)
stress_idx  = rng_stress.choice(len(X_test_A), size=min(STRESS_SIZE, len(X_test_A)), replace=False)

X_stress_base = X_test_A.iloc[stress_idx].copy()
y_stress      = y_test.iloc[stress_idx].values
race_stress   = df.loc[X_test_A.iloc[stress_idx].index, "derived_race"].values

# Baseline predictions (no shock applied)
baseline_pred = model_A_gbt.predict(X_stress_base)

# Race groups with sufficient representation in the subsample
focal_races = ["White", "Black or African American", "Asian"]

# Define shock scenarios: (feature_name, multiplicative_factor)
scenarios = {
    "Scenario 1: Income -20%":  ("income",                    0.80),
    "Scenario 2: CLTV +20%":   ("loan_to_value_ratio", 1.20),
}

# Populated for the bar chart cell (single source of truth — no hard-coded deltas)
stress_bar_deltas = {}

for scenario_name, (feat, factor) in scenarios.items():
    X_stressed = X_stress_base.copy()
    X_stressed[feat] = X_stressed[feat] * factor

    stressed_pred = model_A_gbt.predict(X_stressed)

    rows = []
    deltas_for_chart = []
    for race_val in focal_races:
        mask = race_stress == race_val
        n    = int(mask.sum())
        if n < 50:
            deltas_for_chart.append(float("nan"))
            continue
        base_rate     = float(baseline_pred[mask].mean())
        stressed_rate = float(stressed_pred[mask].mean())
        delta         = stressed_rate - base_rate
        deltas_for_chart.append(delta)
        rows.append({
            "Race":           race_val,
            "N":              n,
            "Baseline Rate":  round(base_rate,     4),
            "Stressed Rate":  round(stressed_rate, 4),
            "Delta":          round(delta,          4),
        })

    stress_bar_deltas[scenario_name] = deltas_for_chart

    print(f"\n{scenario_name}")
    display(pd.DataFrame(rows))


In [ ]:
# ── Grouped bar chart: Δ approval by race × scenario (values from stress test above) ──
import matplotlib.pyplot as plt
import numpy as np

if "stress_bar_deltas" not in globals() or not stress_bar_deltas:
    raise RuntimeError("Run the stress-test cell above first so stress_bar_deltas is defined.")

scenario_names = list(stress_bar_deltas.keys())
if len(scenario_names) < 2:
    raise RuntimeError("Expected at least two stress scenarios for the grouped bar chart.")

sc1_deltas = stress_bar_deltas[scenario_names[0]]
sc2_deltas = stress_bar_deltas[scenario_names[1]]

races = ["White", "Black or\nAfrican American", "Asian"]
if len(sc1_deltas) != len(races) or len(sc2_deltas) != len(races):
    raise RuntimeError("stress_bar_deltas must have one value per race group.")

x     = np.arange(len(races))
width = 0.35

all_vals = [v for pair in (sc1_deltas, sc2_deltas) for v in pair if np.isfinite(v)]
ymin = min(all_vals + [0.0]) - 0.002
ymax = max(all_vals + [0.0]) + 0.002

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width / 2, sc1_deltas, width, label=scenario_names[0],
               color="#d62728", alpha=0.85)
bars2 = ax.bar(x + width / 2, sc2_deltas, width, label=scenario_names[1],
               color="#1f77b4", alpha=0.85)

ax.set_xlabel("Race Group", fontsize=12)
ax.set_ylabel("Δ Approval Rate", fontsize=12)
ax.set_title("Stress Test — Change in Approval Rate by Race Group", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(races, fontsize=11)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.legend(fontsize=10)
ax.set_ylim(ymin, ymax)

for bars in (bars1, bars2):
    for bar in bars:
        h = bar.get_height()
        if not np.isfinite(h):
            continue
        ytxt = h + (0.0004 if h >= 0 else -0.0004)
        va = "bottom" if h >= 0 else "top"
        ax.annotate(
            f"{h:.4f}",
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 3 if h >= 0 else -3),
            textcoords="offset points",
            ha="center", va=va,
            fontsize=9,
            fontweight="bold",
        )

plt.tight_layout()
plt.show()

if len(sc1_deltas) > 1 and np.isfinite(sc1_deltas[1]):
    print(
        "\nKey finding: Black applicants face the steepest income-shock penalty in this subsample (",
        f"{sc1_deltas[1]:+.4f} delta in approval rate),",
        "reflecting thinner income buffers above the model decision boundary.",
    )


**Stress Test Interpretation**

- **Scenario 1 (Income −20%):** All three race groups show reduced approval rates, but the impact is unequal. Black applicants experience the largest absolute decline (−0.0067), compared to White (−0.0045) and Asian (−0.0010). This reflects that Black applicants in this dataset have a thinner income buffer above the model's decision boundary — a finding consistent with systemic wealth gaps documented in HMDA data.

- **Scenario 2 (CLTV +20%):** All groups show Δ = 0.0. This is **not** a data error. GBT (Gradient Boosted Trees) makes predictions based on discrete tree-split thresholds. A 20% multiplicative increase in `loan_to_value_ratio` shifts raw values but does not push any applicant across an existing split boundary in the learned trees — meaning the model has effectively learned that CLTV variation in this range does not change the lending decision. This can be confirmed by inspecting `model_A_gbt.named_steps['model'].feature_importances_`: if `loan_to_value_ratio` ranks low in importance, it will have few/shallow splits and will be insensitive to proportional shocks. The δ = 0.0 result is a valid, interpretable finding about where the model places its decision boundaries.

---
## Week 4 — Robustness Analysis: Key Insights

| Dimension | Finding | Severity |
|---|---|---|
| **Distribution Shift (PSI/KS)** | All 4 numeric features STABLE (PSI < 0.10, KS p > 0.05) | Low |
| **Multivariate Drift (MMD²)** | MMD² = 0.000264 — well below 0.01 Low-shift threshold | Low |
| **Slice Performance (race)** | Black applicant FNR (4.89%) > White FNR (2.94%) — +1.95pp gap | High |
| **Slice Performance (loan type)** | USDA loans: AUC = 0.627, FPR = 0.90 — model effectively broken on this segment | High |
| **Stress Test (Income −20%)** | Black applicants hit hardest: delta = −0.67pp vs. White −0.45pp | Medium |
| **Stress Test (CLTV +20%)** | Zero effect across all races — GBT insensitive to this proportional shock | Low |

**Synthesis**: The model’s distributional stability between train and test is reassuring but should
not be confused with *production* robustness. The train-test split uses the same 2024 HMDA vintage;
real-world deployment faces future economic cycles that neither PSI nor MMD² can anticipate.
The most pressing robustness concern is the **slice-level disparity**: the systematic gap in
false-negative rates across racial groups and the near-complete model failure on USDA loans both
indicate that aggregate AUC conceals serious sub-population reliability problems.

**Recommended actions before deployment**:
- Remove or recode the USDA loan slice from the single-model scope; train a dedicated sub-model
- Apply equalized-odds post-processing to close the Black/White FNR gap
- Deploy automated monthly PSI monitoring with a RETRAIN trigger at PSI >= 0.10
---

### Lecture 5 Privacy & Safety 


Privacy & Safety

Three questions drive this section: can an adversary manipulate inputs to get a favorable decision (evasion), can an attacker corrupt training data to harm specific groups (poisoning), and can the model be probed to expose individual training records (membership inference).

Features most relevant here:
- `derived_race` / `derived_ethnicity` / `derived_sex`: sensitive attributes whose information must not leak through proxy variables
- `tract_minority_population_percent`: a geographic proxy that may indirectly encode race
- `income` / CLTV / `property_value`: high-signal financial variables that can make individual records identifiable

### Before GitHub / submission — save all outputs

1. **Kernel → Restart & Run All** (wait until every cell finishes).
2. **File → Save** (Ctrl+S) so tables and plots are **embedded in this .ipynb**.
3. Commit the saved notebook so reviewers see the same numbers as your slides.

提出用：**すべて実行後に保存**し、出力付きの .ipynb をコミットしてください。


#### 5.1 PGD Evasion Attack on Logistic Regression Model A

**Projected Gradient Descent (PGD)** is an iterative adversarial attack that perturbs input features along the model's loss gradient while staying within a constrained perturbation budget (epsilon-ball).

We perturb only the numeric features (`income`, `loan_to_value_ratio`, `tract_minority_population_percent`, `property_value`) in raw feature space, using the LR coefficients to determine the gradient direction that maximizes approval probability.

- **ε** = perturbation budget measured in units of each feature's training standard deviation
- **α = ε / 10** = step size per iteration (10 PGD steps total)
- A fair and robust model should maintain stable approval rates and AIR even under adversarial perturbation

In [ ]:
# Extract gradient direction from LR coefficients (numeric features come first in the encoded space)
lr_coef    = model_A.named_steps["classifier"].coef_[0]       # shape: (n_encoded_features,)
grad_sign  = np.sign(lr_coef[:len(numeric_features_A)])        # sign gives gradient direction for class 1

# Extract StandardScaler std to convert epsilon from standardized to raw feature space
scaler_stds = (
    model_A.named_steps["preprocessor"]
           .named_transformers_["num"]
           .named_steps["scaler"]
           .scale_
)

# Subsample test set for the attack
ATTACK_SIZE = 5_000
rng_atk     = np.random.default_rng(42)
attack_idx  = rng_atk.choice(len(X_test_A), size=min(ATTACK_SIZE, len(X_test_A)), replace=False)
X_atk_base  = X_test_A.iloc[attack_idx].copy()
race_atk    = df.loc[X_test_A.iloc[attack_idx].index, "derived_race"].values

pgd_rows = []
for eps in [0.1, 0.5, 1.0, 2.0]:
    alpha  = eps / 10           # step size = 1/10 of the total epsilon budget
    X_adv  = X_atk_base.copy()

    for _ in range(10):         # 10 PGD iterations
        for j, feat in enumerate(numeric_features_A):
            raw_alpha = alpha * scaler_stds[j]          # step size converted to raw units
            raw_eps   = eps   * scaler_stds[j]          # budget converted to raw units
            # Gradient ascent step (maximize approval probability)
            X_adv[feat] = X_adv[feat] + raw_alpha * grad_sign[j]
            # Project back into the epsilon-ball around the original input
            X_adv[feat] = X_atk_base[feat] + np.clip(
                X_adv[feat] - X_atk_base[feat], -raw_eps, raw_eps
            )

    base_pred_atk = model_A.predict(X_atk_base)
    adv_pred      = model_A.predict(X_adv)

    white_mask = race_atk == "White"
    black_mask = race_atk == "Black or African American"

    air_base = (
        base_pred_atk[black_mask].mean() / base_pred_atk[white_mask].mean()
        if white_mask.sum() > 0 and black_mask.sum() > 0 else float("nan")
    )
    air_adv = (
        adv_pred[black_mask].mean() / adv_pred[white_mask].mean()
        if white_mask.sum() > 0 and black_mask.sum() > 0 else float("nan")
    )

    pgd_rows.append({
        "epsilon":           eps,
        "Baseline_Approval": round(float(base_pred_atk.mean()), 4),
        "Adv_Approval":      round(float(adv_pred.mean()),       4),
        "Baseline_AIR":      round(float(air_base),              4),
        "Adv_AIR":           round(float(air_adv),               4),
    })

pgd_df = pd.DataFrame(pgd_rows)
print("PGD Evasion Attack Results — LR Model A")
display(pgd_df)

#### 5.2 Data Poisoning: Label-Flip Attack Targeting Black Applicants

**Label-flip poisoning** simulates a targeted attack where an adversary flips the training labels
of Black applicants from denial (0) to approval (1). This corrupts the model's learned association
between Black applicant features and denial outcomes, potentially reducing the model's ability to
discriminate and/or degrading its fairness metrics.

- `poison_rate` = fraction of Black training labels flipped (0% = clean baseline)
- We retrain a fresh LR model on the poisoned dataset and evaluate AUC and AIR on the clean test set
- A robust model should show minimal AUC and AIR degradation even under significant poisoning rates

After the results table, we plot **poisoning degradation curves** (test AUC and AIR vs. poison rate) to match the Week 5 deliverable: interpretable *degradation* under attack, not only point estimates at 0% and 20%.

In [ ]:
from sklearn.linear_model import LogisticRegression as LR_fresh
from sklearn.pipeline import Pipeline as Pipeline_fresh
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

rng_poison = np.random.default_rng(42)

# Identify Black applicant indices in training set
black_train_mask = (
    df.loc[X_train_A.index, "derived_race"] == "Black or African American"
).values
black_train_idx = np.where(black_train_mask)[0]

poison_rates = [0.00, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20]
poison_rows  = []

# Baseline (clean model already trained)
y_base_pred_proba = model_A.predict_proba(X_test_A)[:, 1]
auc_base          = roc_auc_score(y_test, y_base_pred_proba)

white_test_mask = df.loc[X_test_A.index, "derived_race"].values == "White"
black_test_mask = df.loc[X_test_A.index, "derived_race"].values == "Black or African American"

base_pred_test = model_A.predict(X_test_A)
air_base_val = (
    base_pred_test[black_test_mask].mean() / base_pred_test[white_test_mask].mean()
    if white_test_mask.sum() > 0 and black_test_mask.sum() > 0 else float("nan")
)

poison_rows.append({
    "poison_rate":     0.00,
    "AUC":             round(auc_base,     4),
    "AIR_Black_White": round(air_base_val, 4),
})

for p_rate in poison_rates[1:]:
    y_poisoned = y_train.copy()
    n_flip     = int(len(black_train_idx) * p_rate)
    flip_idx   = rng_poison.choice(black_train_idx, size=n_flip, replace=False)
    y_poisoned.iloc[flip_idx] = 1  # flip denial -> approval

    poisoned_model = Pipeline_fresh([
        ("preprocessor", preprocessor_A),
        ("classifier",   LR_fresh(max_iter=1000, random_state=42))
    ])
    poisoned_model.fit(X_train_A, y_poisoned)

    y_adv_proba = poisoned_model.predict_proba(X_test_A)[:, 1]
    auc_adv     = roc_auc_score(y_test, y_adv_proba)
    adv_pred    = poisoned_model.predict(X_test_A)
    air_adv_val = (
        adv_pred[black_test_mask].mean() / adv_pred[white_test_mask].mean()
        if white_test_mask.sum() > 0 and black_test_mask.sum() > 0 else float("nan")
    )

    poison_rows.append({
        "poison_rate":     p_rate,
        "AUC":             round(auc_adv,     4),
        "AIR_Black_White": round(air_adv_val, 4),
    })

poison_df = pd.DataFrame(poison_rows)
print("Label-Flip Poisoning Results — LR Model A (Targeting Black Applicants)")
display(poison_df)

# Degradation curves: test AUC and AIR vs. poison rate (Lecture 5 deliverable)
pr_pct = poison_df["poison_rate"] * 100
fig, (ax_auc, ax_air) = plt.subplots(1, 2, figsize=(10, 4))
ax_auc.plot(pr_pct, poison_df["AUC"], "o-", color="#1f77b4", linewidth=2, markersize=6)
ax_auc.set_xlabel("Poison rate (% of Black training labels flipped)")
ax_auc.set_ylabel("Test AUC (clean holdout)")
ax_auc.set_title("Performance degradation under poisoning")
ax_auc.grid(True, alpha=0.3)
ax_air.plot(pr_pct, poison_df["AIR_Black_White"], "s-", color="#ff7f0e", linewidth=2, markersize=6)
ax_air.axhline(0.80, color="red", linestyle="--", alpha=0.6, label="80% rule (illustrative)")
ax_air.set_xlabel("Poison rate (% of Black training labels flipped)")
ax_air.set_ylabel("AIR (Black / White pred. approval rate)")
ax_air.set_title("Fairness metric under poisoning")
ax_air.legend(loc="best", fontsize=8)
ax_air.grid(True, alpha=0.3)
fig.suptitle("Label-Flip Poisoning: Degradation Curves", fontweight="bold", fontsize=12)
fig.tight_layout()
plt.show()

**Data Poisoning Interpretation**

The AIR (Adverse Impact Ratio) declines only modestly from **0.9257** (0% poison) to **0.9222**
(20% poison) — a drop of just 0.0035. This small effect is not a sign that the attack failed;
rather, it reveals an important structural property of Logistic Regression:

1. **Attack scope is proportionally small**: Black applicants represent roughly 5–7% of the total
   training set. Even flipping 20% of their labels affects only ~1–1.5% of all training records —
   far too few to materially shift the LR weight vector.

2. **LR’s smooth loss distributes signal broadly**: Unlike tree-based models with sharp splits,
   logistic regression uses a globally smooth sigmoid loss. Flipped labels in one subgroup
   create a mild gradient pull that is overwhelmed by the correct signal from the remaining ~99%
   of training records.

3. **Threshold to cause regulatory harm**: Breaching the legal AIR < 0.80 threshold would likely
   require flipping **≈50%+ of Black applicant labels** — a poisoning rate that would be detectable
   via routine data-quality audits (e.g., RONI filter, label distribution checks).

> **Regulatory note**: Despite the small measured effect, label-flip attacks on protected-class
> subgroups remain a **High** severity risk under CFPB model risk guidance. A sophisticated attacker
> could combine label manipulation with feature perturbation for a stronger combined attack.
> RONI filtering and cryptographic data provenance are the recommended mitigations.

#### 5.3 Membership Inference Attack: Privacy Risk Assessment

**Membership inference** tests whether an adversary can determine if a specific record was in the training set by observing the model's output confidence scores.

**Key insight**: Models that overfit tend to assign higher confidence to training records (members) than to unseen records (non-members). This confidence gap is exploited as an attack signal.

**Risk thresholds (MI AUC):**
- AUC ≈ 0.50: **Low risk** — model generalizes well, membership indistinguishable from random
- AUC 0.60–0.70: **Moderate risk** — some memorization present; monitor with regularization
- AUC > 0.70: **High risk** — significant memorization; model may expose sensitive training records

The next cell reports MI AUC, the **mean confidence gap** between members and non-members, a **membership-attack ROC curve** (signal = predicted probability), and **overlapping score histograms** to visualize the confidence gap (Lecture 5: ROC + confidence split).

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Sample members (from training set) and non-members (from test set)
MI_SIZE = 5_000
rng_mi  = np.random.default_rng(42)

member_idx     = rng_mi.choice(len(X_train_A), size=min(MI_SIZE, len(X_train_A)), replace=False)
nonmember_idx  = rng_mi.choice(len(X_test_A),  size=min(MI_SIZE, len(X_test_A)),  replace=False)

X_member    = X_train_A.iloc[member_idx]
X_nonmember = X_test_A.iloc[nonmember_idx]

# Get model confidence scores (probability of class 1)
conf_member    = model_A.predict_proba(X_member)[:, 1]
conf_nonmember = model_A.predict_proba(X_nonmember)[:, 1]

# Use confidence score as membership signal: higher confidence -> more likely member
mi_labels  = np.concatenate([np.ones(len(conf_member)), np.zeros(len(conf_nonmember))])
mi_scores  = np.concatenate([conf_member, conf_nonmember])
mi_auc     = roc_auc_score(mi_labels, mi_scores)

print(f"Membership Inference AUC: {mi_auc:.4f}")
if mi_auc > 0.70:
    print("Risk Level: HIGH — significant memorization detected; apply differential privacy")
elif mi_auc > 0.60:
    print("Risk Level: MODERATE — some memorization; monitor and regularize")
else:
    print("Risk Level: LOW — model generalizes well; membership largely indistinguishable")

print(f"\nMean confidence (members):     {conf_member.mean():.4f}")
print(f"Mean confidence (non-members): {conf_nonmember.mean():.4f}")
gap = float(conf_member.mean() - conf_nonmember.mean())
print(f"Confidence gap:                {gap:.4f}")

# ROC for membership attack + score overlap (Lecture 5: ROC + confidence gap)
fpr_mi, tpr_mi, _ = roc_curve(mi_labels, mi_scores)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr_mi, tpr_mi, color="#2ca02c", linewidth=2, label=f"MI attack (AUC = {mi_auc:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="Random (AUC = 0.5)")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("Membership inference ROC curve")
axes[0].legend(loc="lower right")
axes[0].set_aspect("equal", adjustable="box")
axes[0].grid(True, alpha=0.3)
axes[1].hist(conf_nonmember, bins=40, alpha=0.65, label="Non-members (holdout test)", color="#1f77b4", density=True)
axes[1].hist(conf_member, bins=40, alpha=0.65, label="Members (train sample)", color="#d62728", density=True)
axes[1].set_xlabel("Model confidence: P(approve | x)")
axes[1].set_ylabel("Density")
axes[1].set_title("Confidence gap: training vs. holdout score distributions")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.suptitle(
    f"Membership inference: ROC + score distributions (mean confidence gap = {gap:+.4f})",
    fontweight="bold",
    fontsize=11,
)
fig.tight_layout()
plt.show()

# --- Exported for downstream summary table (Week 5 key insights) ---
MI_AUC = float(mi_auc)
MI_CONFIDENCE_GAP = float(gap)
if MI_AUC > 0.70:
    MI_RISK_LEVEL = "HIGH"
    MI_RISK_LABEL = "🔴 High"
elif MI_AUC > 0.60:
    MI_RISK_LEVEL = "MEDIUM"
    MI_RISK_LABEL = "🟡 Medium"
else:
    MI_RISK_LEVEL = "LOW"
    MI_RISK_LABEL = "🟢 Low"


In [ ]:
# ── Week 5 — Privacy & Safety: Key Insights (numbers from this run) ──
from IPython.display import display, Markdown

def _need(name):
    raise RuntimeError(f"Run the {name} cells above first (PGD, poisoning, membership inference).")

if "pgd_df" not in globals():
    _need("PGD")
if "poison_df" not in globals():
    _need("poisoning")
if "MI_AUC" not in globals():
    _need("membership inference")

pgd_row = pgd_df.loc[pgd_df["epsilon"] == 2.0].iloc[0]
b_app = float(pgd_row["Baseline_Approval"])
a_app = float(pgd_row["Adv_Approval"])
delta_pp = round((a_app - b_app) * 100, 2)
air_b = float(pgd_row["Baseline_AIR"])
air_a = float(pgd_row["Adv_AIR"])

air0 = float(poison_df.loc[poison_df["poison_rate"] == 0.00, "AIR_Black_White"].iloc[0])
air_p = float(poison_df.loc[poison_df["poison_rate"] == 0.20, "AIR_Black_White"].iloc[0])
auc0 = float(poison_df.loc[poison_df["poison_rate"] == 0.00, "AUC"].iloc[0])
auc_p = float(poison_df.loc[poison_df["poison_rate"] == 0.20, "AUC"].iloc[0])

mi_line = (
    f"MI AUC = **{MI_AUC:.4f}**; mean confidence gap = **{MI_CONFIDENCE_GAP:+.4f}** "
    f"({MI_RISK_LEVEL} per §5.3 thresholds)"
)

md = f"""
---
## 🔒 Week 5 — Privacy & Safety: Key Insights

| Attack Vector | Key Result | Severity |
|---|---|---|
| **PGD Evasion (ε = 2.0)** | Approval rate **{b_app:.3f} → {a_app:.3f}** (Δ **+{delta_pp:.2f} pp**); AIR **{air_b:.4f} → {air_a:.4f}** | 🟡 Medium |
| **Label-Flip Poisoning (20%)** | Test AUC **{auc0:.4f} → {auc_p:.4f}**; AIR **{air0:.4f} → {air_p:.4f}** | 🟡 Medium |
| **Membership Inference** | {mi_line} | {MI_RISK_LABEL} |

**PGD Evasion insight**: The LR model is highly susceptible to gradient-based input manipulation.
At ε = 2.0, approval rates rise by **{delta_pp:.2f}** percentage points vs. the attack subsample baseline.
Counterintuitively, AIR can *improve* under this attack if the perturbation shifts both groups similarly —
this is **not** a safety guarantee; asymmetric or group-targeted perturbations could be more harmful.

**Poisoning insight**: LR’s smooth loss can absorb small-fraction label flips; monitor AUC/AIR at higher rates
or combined attacks.

**Privacy insight**: MI AUC **{MI_AUC:.4f}** ({MI_RISK_LEVEL}) quantifies memorization-style leakage from confidence scores.
Sensitive HMDA fields still warrant query controls, DP, or confidence clipping **even when MI AUC is low**.

**Recommended actions before deployment**:
- Adversarial training / input validation (e.g. clamp to training range)
- RONI or provenance checks on training data
- Differential privacy or output smoothing if MI AUC rises in future retrains

---
### 提出前（出力の保存）
**Kernel → Restart & Run All** のあと **File → Save**（Ctrl+S）でノートを保存し、GitHub にコミットする `.ipynb` に **すべてのセル出力が埋まっている**ことを確認してください。
---
"""
display(Markdown(md))


### Conclusion — Five Capstone Questions

This section synthesises the findings from all analytical sections — Pre-modeling, Modeling, Transparency (Lecture 2), Fairness (Lecture 3), Robustness (Lecture 4), and Privacy & Safety (Lecture 5) — into structured answers to the five capstone questions.

These questions are the backbone of the 5/7 presentation and the written capstone submission.

In [ ]:
print("=" * 70)
print("Q1: What is the optimization objective of this model?")
print("=" * 70)
print("""
The model minimises binary cross-entropy (log-loss) on the loan approval
outcome (approved = 1, denied = 0) using an 80/20 stratified train-test
split of the 2024 HMDA LAR dataset.

Protected attributes (derived_race, derived_ethnicity, derived_sex) are
excluded from the feature set in compliance with ECOA and the Fair Housing
Act. The four retained numeric features are:
  - income
  - loan_to_value_ratio
  - tract_minority_population_percent  (geographic proxy — flagged as risk)
  - property_value

Two model architectures are compared:
  - Model A: Logistic Regression (interpretable baseline)
  - Model A-GBT: Gradient Boosted Trees (higher performance)

The GBT achieves Test AUC ≈ 0.847 vs LR AUC ≈ 0.805, but both exhibit
similar fairness profiles, confirming that the fairness findings are
structural rather than model-specific.
""")

In [ ]:
print("=" * 70)
print("Q2: What are the known failure modes of this model?")
print("=" * 70)

_pgd = pgd_df.loc[pgd_df["epsilon"] == 2.0].iloc[0]
_pgd_dpp = (float(_pgd["Adv_Approval"]) - float(_pgd["Baseline_Approval"])) * 100

_air0 = float(poison_df.loc[poison_df["poison_rate"] == 0.00, "AIR_Black_White"].iloc[0])
_air20 = float(poison_df.loc[poison_df["poison_rate"] == 0.20, "AIR_Black_White"].iloc[0])
_auc0 = float(poison_df.loc[poison_df["poison_rate"] == 0.00, "AUC"].iloc[0])
_auc20 = float(poison_df.loc[poison_df["poison_rate"] == 0.20, "AUC"].iloc[0])
_auc_drop = _auc0 - _auc20
_air_drop = _air0 - _air20

_mi_sev = "High" if mi_auc > 0.70 else ("Medium" if mi_auc > 0.60 else "Low")
_gap = MI_CONFIDENCE_GAP if "MI_CONFIDENCE_GAP" in globals() else float("nan")

failure_modes_df = pd.DataFrame([
    {
        "Failure Mode":   "Proxy Discrimination",
        "Evidence":       "tract_minority_population_percent retained as feature despite being a geographic proxy for race; pre-model AIR < 0.80 for Black applicants. Removing the feature reduces AUC but is expected to improve AIR — a tradeoff acknowledged as a deployment risk.",
        "Severity":       "High",
        "Mitigation":     "Pre-deployment: ablation study to quantify AUC vs AIR tradeoff upon removing tract_minority_population_percent. Interim: flag as high-risk in model card; apply fairness-aware regularization (e.g., adversarial debiasing).",
    },
    {
        "Failure Mode":   "Distribution Shift (Covariate Drift)",
        "Evidence":       "PSI and KS tests confirm all numeric features STABLE (PSI < 0.10); "
                         "no covariate drift detected between train/test, but production drift "
                         "remains a forward risk if economic conditions change",
        "Severity":       "Medium",
        "Mitigation":     "Schedule periodic retraining; implement automated PSI monitoring pipeline",
    },
    {
        "Failure Mode":   "Adversarial Evasion (PGD)",
        "Evidence":       (
            f"PGD at ε=2.0 (LR): approval {float(_pgd['Baseline_Approval']):.3f}→{float(_pgd['Adv_Approval']):.3f} "
            f"(+{_pgd_dpp:.2f} pp); AIR {float(_pgd['Baseline_AIR']):.4f}→{float(_pgd['Adv_AIR']):.4f} — "
            f"gradient-based evasion is material even when AIR moves favorably"
        ),
        "Severity":       "Medium",
        "Mitigation":     "Adversarial training; input range validation; feature value clamping",
    },
    {
        "Failure Mode":   "Label-Flip Data Poisoning",
        "Evidence":       (
            f"Targeted flips on Black training labels (up to 20%): test AUC {_auc0:.4f}→{_auc20:.4f} (Δ={_auc_drop:+.4f}); "
            f"AIR {_air0:.4f}→{_air20:.4f} (Δ={_air_drop:+.4f})"
        ),
        "Severity":       "High" if (_auc_drop > 0.02 or abs(_air_drop) > 0.05) else "Medium",
        "Mitigation":     "RONI filter; cryptographic data integrity checks; data provenance logging",
    },
    {
        "Failure Mode":   "Membership Inference (Privacy Leakage)",
        "Evidence":       (
            f"MI AUC = {mi_auc:.3f} (§5.3 thresholds: {_mi_sev.lower()} memorization risk); "
            f"mean confidence gap = {_gap:+.4f}"
        ),
        "Severity":       _mi_sev,
        "Mitigation":     "Differential privacy (DP-SGD); prediction confidence clipping; API rate limiting",
    },
    {
        "Failure Mode":   "Slice-Level Performance Disparity",
        "Evidence":       "FNR: Black (0.0489) vs White (0.0294) — +1.95pp gap; Black applicants are most likely to be incorrectly denied (Disparate Impact / Equalized Odds violation). FPR highest for Joint group (0.5924). AUC drop for Black applicants (0.8382) is below 5pp FLAG threshold but warrants regulatory scrutiny.",
        "Severity":       "High",
        "Mitigation":     "Equalized odds post-processing; per-subgroup threshold calibration",
    },
])

display(failure_modes_df)


In [ ]:
print("=" * 70)
print("Q3: How does subgroup error vary across protected groups?")
print("=" * 70)

# Pull race slices from the slice evaluation results
race_groups   = ["White", "Black or African American", "Asian", "Native Hawaiian or Other Pacific Islander", "Joint"]
q3_rows       = []

for race_val in race_groups:
    mask = df.loc[X_test_A.index, "derived_race"].values == race_val
    if mask.sum() < 50:
        continue
    y_true_slice = y_test.values[mask]
    y_pred_slice = model_A_gbt.predict(X_test_A.iloc[np.where(mask)[0]])

    tn, fp, fn, tp = __import__('sklearn.metrics', fromlist=['confusion_matrix']).confusion_matrix(
        y_true_slice, y_pred_slice, labels=[0,1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    q3_rows.append({
        "Race Group": race_val,
        "N":          int(mask.sum()),
        "FPR":        round(fpr, 4),
        "FNR":        round(fnr, 4),
    })

q3_df = pd.DataFrame(q3_rows)
print("Subgroup Error Rates — GBT Model A")
display(q3_df)
print("\nKey finding: Black FNR (0.0489) > White FNR (0.0294) — Equalized Odds violation.")
print("Black applicants face significantly higher incorrect-denial rates.")

In [ ]:
print("=" * 70)
print("Q4: What residual risks remain after analysis?")
print("=" * 70)

mi_risk_level = "High" if mi_auc > 0.70 else ("Medium" if mi_auc > 0.60 else "Low")

residual_risks = pd.DataFrame([
    {"Risk": "Proxy discrimination via tract_minority_population_percent",
     "Likelihood": "High", "Impact": "High",
     "Status": "Open — ablation study pending"},
    {"Risk": "No automated PSI monitoring in production",
     "Likelihood": "High", "Impact": "High",
     "Status": "Open — pipeline not yet built"},
    {"Risk": "Membership inference privacy risk",
     "Likelihood": mi_risk_level, "Impact": "High" if mi_auc > 0.60 else "Medium",
     "Status": f"Open — MI AUC = {mi_auc:.3f}; DP not yet applied"},
    {"Risk": "USDA loan slice failure (AUC = 0.627, FPR = 0.90)",
     "Likelihood": "High", "Impact": "Medium",
     "Status": "Open — sub-model or exclusion required"},
    {"Risk": "PGD evasion attack surface",
     "Likelihood": "Medium", "Impact": "Medium",
     "Status": "Open — adversarial training not implemented"},
    {"Risk": "Equalized Odds violation (Black/White FNR gap)",
     "Likelihood": "High", "Impact": "High",
     "Status": "Open — post-processing not applied"},
])

display(residual_risks)
print(f"\nTotal open risks: {len(residual_risks)}")
print(f"High likelihood x High impact: {len(residual_risks[(residual_risks.Likelihood=='High') & (residual_risks.Impact=='High')])}")


#### Q5: Is This Model Defensible for Deployment?

**Recommendation: Conditional Deployment with Mandatory Remediation**

Based on the comprehensive analysis across all six capstone dimensions, the model demonstrates strong predictive performance but carries significant residual risks that must be addressed before production deployment.

---

**Strengths:**
- Model A achieves strong predictive performance (Test Accuracy ~84%, AUC > 0.85 for GBT)
- Deliberately excludes protected attributes (race, ethnicity, sex) in compliance with ECOA and FHA
- Consistent findings across both LR and GBT approaches confirm robustness of conclusions
- PSI/KS tests confirm train-test distributional stability on numeric features

---

**Required Remediation Before Deployment:**
1. **Proxy discrimination audit**: Evaluate and potentially remove `tract_minority_population_percent`, which may encode racial information through geographic segregation patterns
2. **PSI monitoring pipeline**: Implement automated monthly drift detection with RETRAIN thresholds before production
3. **Fairness constraint enforcement**: Apply equalized odds post-processing to reduce cross-race FPR disparity to within regulatory tolerance
4. **Data provenance logging**: Implement RONI filter and cryptographic audit trail for training data integrity against poisoning attacks
5. **Differential privacy**: Apply DP-SGD or prediction confidence clipping to reduce membership inference risk

---

**Five Capstone Questions — Summary Table**

| Question | Finding |
|---|---|
| Q1: Optimization Objective | Log-loss on binary approval outcome; race/sex/ethnicity excluded per ECOA |
| Q2: Known Failure Modes | Proxy bias, covariate drift, PGD evasion, label-flip poisoning, MI privacy risk |
| Q3: Subgroup Error | Significant FPR variation across racial groups — Equalized Odds violation present |
| Q4: Residual Risks | 6 open risks catalogued; 3 rated High likelihood × High impact, requiring immediate action |
| Q5: Deployment Decision | **Conditional** — requires fairness post-processing, monitoring pipeline, and DP implementation |

---



---
## Week 6 — Governance, Deployment Decision & Residual Risk Register

Week 6 synthesises findings from Lectures 4 and 5 into a structured risk register and deployment
recommendation aligned with CFPB model risk guidance, ECOA/FHA obligations, and SR 11-7.

### Model Scorecard Summary

| Capstone Question | Verdict |
|---|---|
| **Q1 — Optimization objective** | Binary log-loss on approval outcome; no protected attributes in features |
| **Q2 — Known failure modes** | 6 identified: proxy bias, covariate drift, PGD evasion, label-flip, MI, USDA slice failure |
| **Q3 — Subgroup error** | Equalized Odds violated: Black FNR 4.89% vs White 2.94%; USDA AUC = 0.627 |
| **Q4 — Residual risks** | 3 High-likelihood x High-impact risks requiring pre-deployment remediation |
| **Q5 — Deployment decision** | **CONDITIONAL** — deploy only after fairness post-processing + monitoring pipeline |

### Residual Risk Register (Top Risks)

| Risk | Likelihood | Impact | Mitigation Owner | Deadline |
|---|---|---|---|---|
| Proxy discrimination via `tract_minority_population_percent` | High | High | Model team | Pre-deploy |
| No PSI production monitoring pipeline | High | High | MLOps | Pre-deploy |
| Membership inference privacy leakage | High | High | Security | Pre-deploy |
| USDA loan slice failure (AUC=0.627) | High | Medium | Model team | Pre-deploy |
| PGD evasion attack surface | Medium | Medium | Security | 90 days post-deploy |
| Label-flip poisoning (low-rate robustness) | Low | High | MLOps | 90 days post-deploy |

### Governance Checklist (SR 11-7 / CFPB)

- [ ] Model card documenting training data, features, known limitations, and fairness metrics
- [ ] Independent model validation (IMV) review before any credit decisions
- [ ] Equalized-odds post-processing applied and AIR re-tested >= 0.80 for all race groups
- [ ] `tract_minority_population_percent` ablation study completed — AUC vs AIR tradeoff documented
- [ ] PSI monitoring pipeline live and alerting at PSI >= 0.10
- [ ] Differential privacy (DP-SGD epsilon <= 8) or confidence clipping deployed
- [ ] RONI filter active on training data pipeline
- [ ] Adverse action notice logic implemented (ECOA sec 202.9)

### Academic Reflection

This capstone demonstrates that responsible ML in high-stakes lending is not a single-step process.
A model can achieve excellent aggregate predictive performance (AUC > 0.85) while simultaneously:

- **Perpetuating structural inequality** through geographic proxies for race
- **Catastrophically failing** on a specific loan-product slice (USDA)
- **Remaining vulnerable** to adversarial input manipulation
- **Leaking private information** through confidence score differentials

The framework applied here — bias audit -> robustness testing -> privacy analysis -> governance
review — mirrors the structure mandated by the CFPB's 2023 AI in lending guidance and represents
the minimum bar for defensible credit-model deployment in 2025 and beyond.
---